# NB-Step2 · Re-extract Annotated Frames Through NB-01 Pipeline
**Pipeline position:** Step 2 of 9 — runs after NB-Step1 manifest, before NB-Step3 coordinate remap.

### Purpose
The original CVAT annotations were made on raw frames that never passed through the NB-01
preprocessing chain (ROI crop → grayscale → 0.5× bicubic → CLAHE).  This mismatch is the
root cause of the 13.7 % / 7.4 % neural network coverage reported in the paper.

This notebook closes that gap by re-extracting the exact 147 annotated frames from the two
source `.mp4` files and running them through the identical NB-01 chain.  The outputs are
the temporally-aligned preprocessed ground truth frames that NB-Step3 will pair with the
remapped COCO annotations.

### Inputs
| File | Location |
|------|----------|
| `harvest_manifest.json` | `anot_pool/` — frame indices + video paths for all 147 frames |
| `VID_20260429_124526.mp4` | `campaigns/setup_A/run_002_Q20/` |
| `VID_20260429_125119.mp4` | `campaigns/setup_A/run_003_Q30/` |

### Outputs (all written to `anot_pool/preproc_frames_147/`)
| File | Description |
|------|-------------|
| `VID_<...>_frame<idx>.png` | 147 preprocessed grayscale PNGs (1080×1475 px) |
| `preproc_manifest.json` | Frame-level extraction log consumed by NB-Step3 |


In [ ]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import cv2
import numpy as np
import json, os, time
from pathlib import Path
from datetime import datetime

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f"✓ Imports complete  |  OpenCV {cv2.__version__}  |  NumPy {np.__version__}")


In [ ]:
# ── Cell 2 · Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Drive mounted")


In [ ]:
# ── Cell 3 · Static Configuration ───────────────────────────────────────────
# HUMAN-EDITED section — only these paths should ever need changing.

ANNOT_BASE = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool"

HARVEST_MANIFEST = f"{ANNOT_BASE}/harvest_manifest.json"
OUTPUT_DIR       = f"{ANNOT_BASE}/preproc_frames_147"

# NB-01 preprocessing parameters — must stay identical to harvest notebook.
# These are also stored inside harvest_manifest.json and validated in Cell 4.
PREPROC = {
    "downscale_factor": 0.5,
    "clahe_clip_limit": 2.0,
    "clahe_tile_grid":  (4, 4),
    "interpolation":    cv2.INTER_CUBIC,
}

PNG_QUALITY = [cv2.IMWRITE_PNG_COMPRESSION, 1]   # fast write, lossless

os.makedirs(OUTPUT_DIR, exist_ok=True)
STEP2_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

print("✓ Cell 3 — configuration loaded")
print(f"  Harvest manifest : {HARVEST_MANIFEST}")
print(f"  Output dir       : {OUTPUT_DIR}")


In [ ]:
# ── Cell 4 · Load Harvest Manifest ───────────────────────────────────────────
# Reads harvest_manifest.json and builds a flat list of extraction tasks.
# Also cross-validates the ROI stored in the manifest against PREPROC.

if not os.path.exists(HARVEST_MANIFEST):
    raise FileNotFoundError(
        f"Harvest manifest not found:\n  {HARVEST_MANIFEST}\n"
        "Check Drive is mounted and ANNOT_BASE in Cell 3 is correct."
    )

with open(HARVEST_MANIFEST) as f:
    hm = json.load(f)

# ── Validate ROI from manifest ────────────────────────────────────────────────
MANIFEST_ROI = hm["roi"]   # authoritative — read from the file that created images_29
PREPROC["roi"] = MANIFEST_ROI
print(f"  ROI from manifest: {MANIFEST_ROI}  (loaded into PREPROC)")

x0, y0, x1, y1 = MANIFEST_ROI
ROI_W = x1 - x0
ROI_H = y1 - y0
DS    = PREPROC["downscale_factor"]
PREPROC_W = int(ROI_W * DS)
PREPROC_H = int(ROI_H * DS)

# ── Build flat task list ──────────────────────────────────────────────────────
# Each task: {video_path, frame_idx, harvest_filename, run_label, score}
tasks = []
for video_entry in hm["videos"]:
    if video_entry["n_saved"] == 0:
        continue
    for rec in video_entry["frames"]:
        stem    = Path(rec["filename"]).stem      # e.g. VID_20260429_124526_frame00120
        out_png = os.path.join(OUTPUT_DIR, f"{stem}.png")
        tasks.append({
            "run_label":        video_entry["run_label"],
            "video_path":       video_entry["video_path"],
            "frame_idx":        rec["frame_idx"],
            "harvest_score":    rec["score"],
            "harvest_filename": rec["filename"],
            "stem":             stem,
            "output_png":       out_png,
        })

# ── Group by video for efficient sequential seeking ───────────────────────────
from collections import defaultdict
by_video = defaultdict(list)
for t in tasks:
    by_video[t["video_path"]].append(t)
for vpath in by_video:
    by_video[vpath].sort(key=lambda x: x["frame_idx"])

# ── Report ────────────────────────────────────────────────────────────────────
W = 66
print("=" * W)
print("  Cell 4 — Harvest Manifest Loaded".center(W))
print("=" * W)
print(f"  Total tasks      : {len(tasks)}")
print(f"  Source videos    : {len(by_video)}")
print(f"  ROI (original)   : {MANIFEST_ROI}  →  {ROI_W}×{ROI_H} px")
print(f"  Preprocessed     : {PREPROC_W}×{PREPROC_H} px")
print()
for vpath, vtasks in sorted(by_video.items()):
    run  = vtasks[0]["run_label"]
    idxs = [t["frame_idx"] for t in vtasks]
    print(f"  {run}  —  {len(vtasks)} frames")
    print(f"    Video  : {os.path.basename(vpath)}")
    print(f"    Exists : {'✓' if os.path.exists(vpath) else '✗  FILE NOT FOUND'}")
    print(f"    Indices: {idxs[0]}  →  {idxs[-1]}  ({len(idxs)} frames)")
    print()

missing = [vp for vp in by_video if not os.path.exists(vp)]
if missing:
    raise FileNotFoundError(
        f"Source video(s) not found — check Drive paths:\n"
        + "\n".join(f"  {p}" for p in missing)
    )
print("✓ All source videos found — proceeding to Cell 5")


In [ ]:
# ── Cell 5 · Preprocessing Function ─────────────────────────────────────────
# Exact NB-01 chain: ROI crop → grayscale → 0.5× bicubic → CLAHE.
# Takes a raw BGR frame from cv2.VideoCapture and returns an 8-bit grayscale PNG-ready array.

def preprocess_frame(frame_bgr: np.ndarray, preproc: dict) -> np.ndarray:
    x0, y0, x1, y1 = preproc["roi"]
    h, w   = frame_bgr.shape[:2]
    crop   = frame_bgr[max(0,y0):min(h,y1), max(0,x0):min(w,x1)]
    gray   = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop.copy()
    nw     = int(gray.shape[1] * preproc["downscale_factor"])
    nh     = int(gray.shape[0] * preproc["downscale_factor"])
    small  = cv2.resize(gray, (nw, nh), interpolation=preproc["interpolation"])
    clahe  = cv2.createCLAHE(
        clipLimit    = preproc["clahe_clip_limit"],
        tileGridSize = preproc["clahe_tile_grid"],
    )
    return clahe.apply(small)


def verify_output(png_path: str, expected_w: int, expected_h: int) -> str:
    """Lightweight post-write check. Returns 'ok' or a problem description."""
    if not os.path.exists(png_path):
        return "file not written"
    img = cv2.imread(png_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return "unreadable after write"
    h, w = img.shape
    if w != expected_w or h != expected_h:
        return f"size mismatch: got {w}×{h}, expected {expected_w}×{expected_h}"
    if img.mean() < 1.0:
        return "blank frame"
    return "ok"


print("✓ Cell 5 — preprocessing function defined")
print(f"  Output frame size: {PREPROC_W}×{PREPROC_H} px  (width×height)")


In [ ]:
# ── Cell 6 · Extract and Save Preprocessed Frames ───────────────────────────
# Groups tasks by video so each .mp4 is opened exactly once.
# Seeks directly to each frame_idx — no sequential scan.

extraction_log = []   # one dict per frame, written to preproc_manifest.json in Cell 8
n_ok, n_fail   = 0, 0

for vpath, vtasks in sorted(by_video.items()):
    run_label = vtasks[0]["run_label"]
    print(f"\n  Opening  {run_label}  ({os.path.basename(vpath)})")
    print(f"  Frames to extract: {len(vtasks)}")

    cap = cv2.VideoCapture(vpath)
    if not cap.isOpened():
        print(f"  ✗  Cannot open video — skipping {len(vtasks)} frames")
        for t in vtasks:
            extraction_log.append({**t, "status": "video_open_failed", "verified": "fail"})
            n_fail += 1
        continue

    t0 = time.time()
    for i, task in enumerate(vtasks):
        idx = task["frame_idx"]
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            status = "seek_failed"
            verified = "fail"
            n_fail += 1
            print(f"  ✗  frame {idx:>6}  seek failed")
        else:
            proc     = preprocess_frame(frame, PREPROC)
            cv2.imwrite(task["output_png"], proc, PNG_QUALITY)
            verified = verify_output(task["output_png"], PREPROC_W, PREPROC_H)
            status   = "ok" if verified == "ok" else "write_error"
            if verified == "ok":
                n_ok += 1
            else:
                n_fail += 1
                print(f"  ⚠  frame {idx:>6}  {verified}")

        extraction_log.append({
            "run_label":        task["run_label"],
            "video_path":       task["video_path"],
            "frame_idx":        idx,
            "harvest_score":    task["harvest_score"],
            "harvest_filename": task["harvest_filename"],
            "output_png":       task["output_png"],
            "stem":             task["stem"],
            "status":           status,
            "verified":         verified,
        })

        # Progress every 25 frames
        if (i + 1) % 25 == 0 or (i + 1) == len(vtasks):
            elapsed = time.time() - t0
            print(f"  {i+1:>3}/{len(vtasks)}  extracted  ({elapsed:.1f}s)")

    cap.release()

print(f"\n{'='*50}")
print(f"  Extracted : {n_ok} / {len(tasks)} frames")
if n_fail:
    print(f"  ✗ Failed  : {n_fail} frames — check log in Cell 8")
else:
    print(f"  ✓ Zero failures")


In [ ]:
# ── Cell 7 · Visual QA Panel ─────────────────────────────────────────────────
# Displays a sample of preprocessed PNGs side-by-side with their harvest scores.
# Annotated frames with low scores are flagged in orange.

import random

ok_frames = [r for r in extraction_log if r["verified"] == "ok"]
sample    = sorted(random.sample(ok_frames, min(20, len(ok_frames))),
                   key=lambda x: x["frame_idx"])

n_cols = 5
n_rows = (len(sample) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.8, n_rows * 4.0))
axes = np.array(axes).flatten()

for i, rec in enumerate(sample):
    img   = cv2.imread(rec["output_png"], cv2.IMREAD_GRAYSCALE)
    score = rec["harvest_score"]
    color = "orange" if score < 1.0 else "lime"
    axes[i].imshow(img, cmap="gray", aspect="auto")
    axes[i].set_title(
        f"{rec['run_label']}\nf={rec['frame_idx']}  s={score:.2f}",
        fontsize=7, color=color, pad=2
    )
    axes[i].axis("off")

for j in range(len(sample), len(axes)):
    axes[j].axis("off")

fig.suptitle(
    f"NB-Step2 QA — {len(sample)} sample preprocessed frames  "
    f"(orange = harvest score < 1.0)",
    fontsize=10, fontweight="bold"
)
plt.tight_layout(pad=0.4)
qa_path = os.path.join(OUTPUT_DIR, f"qa_preproc_{STEP2_TS}.png")
plt.savefig(qa_path, dpi=110, bbox_inches="tight")
plt.show()
plt.close()
print(f"✓ QA panel saved: {qa_path}")


In [ ]:
# ── Cell 8 · Export preproc_manifest.json ───────────────────────────────────
# This file is the direct input to NB-Step3 (coordinate remap).
# For each frame it records: stem, frame_idx, output_png, run_label, status.
# NB-Step3 uses stem to match against COCO image filenames.

preproc_manifest = {
    "schema_version": "1.0",
    "generated_at":   STEP2_TS,
    "source_manifest": HARVEST_MANIFEST,
    "output_dir":     OUTPUT_DIR,
    "roi":            MANIFEST_ROI,
    "preproc_w":      PREPROC_W,
    "preproc_h":      PREPROC_H,
    "downscale":      PREPROC["downscale_factor"],
    "n_total":        len(extraction_log),
    "n_ok":           n_ok,
    "n_failed":       n_fail,
    "frames":         extraction_log,
}

out_path = os.path.join(OUTPUT_DIR, f"preproc_manifest_{STEP2_TS}.json")
with open(out_path, "w") as f:
    json.dump(preproc_manifest, f, indent=2)
print(f"✓ preproc_manifest saved: {out_path}")

# ── Summary ───────────────────────────────────────────────────────────────────
W = 66
print()
print("=" * W)
print("  NB-Step2 · SUMMARY".center(W))
print("=" * W)
print(f"  Frames extracted  : {n_ok} / {len(tasks)}")
print(f"  Output size       : {PREPROC_W}×{PREPROC_H} px  (width×height)")
print(f"  Output directory  : {OUTPUT_DIR}")
print()

by_run_log = defaultdict(lambda: {"ok": 0, "fail": 0})
for r in extraction_log:
    key = "ok" if r["verified"] == "ok" else "fail"
    by_run_log[r["run_label"]][key] += 1

print(f"  {'Run label':<20} {'OK':>5} {'Fail':>5}")
print(f"  {'─'*20} {'─'*5} {'─'*5}")
for run, counts in sorted(by_run_log.items()):
    print(f"  {run:<20} {counts['ok']:>5} {counts['fail']:>5}")

if n_fail > 0:
    print()
    print("  ✗ Failed frames:")
    for r in extraction_log:
        if r["verified"] != "ok":
            print(f"    {r['run_label']}  f={r['frame_idx']}  →  {r['status']}")

print()
print("  ✓ Step 2 complete.")
print("  Next → NB-Step3: remap COCO annotation coordinates to preprocessed space.")
print(f"  Key input: {out_path}")
print("=" * W)


In [ ]:
import json

COCO = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool/in_df_147_01.json"

with open(COCO) as f:
    coco = json.load(f)

print("Top-level keys:", list(coco.keys()))
print(f"\nimages    : {len(coco.get('images', []))}")
print(f"annotations: {len(coco.get('annotations', []))}")
print(f"categories : {coco.get('categories', [])}")

print("\nFirst image record:")
print(json.dumps(coco['images'][0], indent=2))

print("\nFirst annotation record:")
print(json.dumps(coco['annotations'][0], indent=2))